In [6]:
import psycopg2
from psycopg2.extras import RealDictCursor
import sys

In [11]:
class RestaurantMenuDB:
    def __init__(self):
        self.conn = None
        self.connect()
        self.init_db()
    
    def connect(self):
        """Установка соединения с базой данных"""
        try:
            self.conn = psycopg2.connect(
                host="localhost", #введите свой хост
                database="restaurant_menu_1", #введите свою БД
                user="postgres", #введите свое имя пользователя
                password="****", #введите свой пароль
                port="5432" #введите свой порт
            )
            print("Успешное подключение к базе данных")
        except Exception as e:
            print(f"Ошибка подключения к базе данных: {e}")
            sys.exit(1)
    
    def init_db(self):
        """Инициализация базы данных и заполнение начальными данными"""
        try:
            with self.conn.cursor() as cur:
                # Создание таблицы категорий
                cur.execute("""
                    CREATE TABLE IF NOT EXISTS category (
                        id INT GENERATED ALWAYS AS IDENTITY PRIMARY KEY,
                        name TEXT NOT NULL UNIQUE
                    )
                """)
                
                # Создание таблицы блюд
                cur.execute("""
                    CREATE TABLE IF NOT EXISTS dish (
                        id INT GENERATED ALWAYS AS IDENTITY PRIMARY KEY,
                        title TEXT NOT NULL,
                        price NUMERIC(8,2) CHECK (price > 0),
                        category_id INT,
                        FOREIGN KEY (category_id) REFERENCES category(id) ON DELETE SET NULL
                    )
                """)
                
                # Заполнение категорий
                categories = ['Супы', 'Салаты', 'Горячее', 'Десерты', 'Напитки']
                for category in categories:
                    cur.execute(
                        "INSERT INTO category (name) VALUES (%s) ON CONFLICT (name) DO NOTHING",
                        (category,)
                    )
                
                # Заполнение блюд (если таблица пуста)
                cur.execute("SELECT COUNT(*) FROM dish")
                if cur.fetchone()[0] == 0:
                    dishes = [
                        ('Борщ', 350.00, 1),
                        ('Окрошка', 280.00, 1),
                        ('Греческий салат', 320.00, 2),
                        ('Цезарь', 380.00, 2),
                        ('Стейк рибай', 1200.00, 3),
                        ('Курица гриль', 450.00, 3),
                        ('Тирамису', 280.00, 4),
                        ('Чизкейк', 320.00, 4),
                        ('Кофе латте', 180.00, 5),
                        ('Сок апельсиновый', 120.00, 5),
                        ('Куриный суп', 290.00, 1),
                        ('Оливье', 270.00, 2)
                    ]
                    
                    for dish in dishes:
                        cur.execute(
                            "INSERT INTO dish (title, price, category_id) VALUES (%s, %s, %s)",
                            dish
                        )
                
                self.conn.commit()
                print("База данных инициализирована успешно")
                
        except Exception as e:
            print(f"Ошибка инициализации базы данных: {e}")
            self.conn.rollback()
    
    def show_all_menu(self):
        """Показать всё меню"""
        try:
            with self.conn.cursor(cursor_factory=RealDictCursor) as cur:
                cur.execute("""
                    SELECT d.title, d.price, c.name as category
                    FROM dish d
                    LEFT JOIN category c ON d.category_id = c.id
                    ORDER BY c.name, d.title
                """)
                dishes = cur.fetchall()
                
                if not dishes:
                    print("Меню пусто")
                    return
                
                print("\n--- ВСЁ МЕНЮ ---")
                for dish in dishes:
                    print(f"{dish['title']} - {dish['price']:.2f} руб. - {dish['category']}")
                print()
                
        except Exception as e:
            print(f"Ошибка при получении меню: {e}")
    
    def show_dishes_in_price_range(self):
        """Показать блюда в ценовом диапазоне"""
        try:
            min_price = float(input("Введите минимальную цену: "))
            max_price = float(input("Введите максимальную цену: "))
            
            with self.conn.cursor(cursor_factory=RealDictCursor) as cur:
                cur.execute("""
                    SELECT d.title, d.price, c.name as category
                    FROM dish d
                    LEFT JOIN category c ON d.category_id = c.id
                    WHERE d.price BETWEEN %s AND %s
                    ORDER BY d.price
                """, (min_price, max_price))
                
                dishes = cur.fetchall()
                
                if not dishes:
                    print("Блюд в указанном диапазоне цен не найдено")
                    return
                
                print(f"\n--- БЛЮДА В ДИАПАЗОНЕ ЦЕН {min_price}-{max_price} руб. ---")
                for dish in dishes:
                    print(f"{dish['title']} - {dish['price']:.2f} руб. - {dish['category']}")
                print()
                
        except ValueError:
            print("Ошибка: введите корректные числовые значения для цен")
        except Exception as e:
            print(f"Ошибка при поиске по цене: {e}")
    
    def search_by_prefix(self):
        """Поиск по началу названия"""
        try:
            prefix = input("Введите начало названия блюда: ").strip()
            
            if not prefix:
                print("Ошибка: введите непустую строку для поиска")
                return
            
            with self.conn.cursor(cursor_factory=RealDictCursor) as cur:
                cur.execute("""
                    SELECT d.title, d.price, c.name as category
                    FROM dish d
                    LEFT JOIN category c ON d.category_id = c.id
                    WHERE LOWER(d.title) LIKE LOWER(%s)
                    ORDER BY d.title
                """, (f"{prefix}%",))
                
                dishes = cur.fetchall()
                
                if not dishes:
                    print("Блюд с таким началом названия не найдено")
                    return
                
                print(f"\n--- РЕЗУЛЬТАТЫ ПОИСКА ПО '{prefix}' ---")
                for dish in dishes:
                    print(f"{dish['title']} - {dish['price']:.2f} руб. - {dish['category']}")
                print()
                
        except Exception as e:
            print(f"Ошибка при поиске: {e}")
    
    def show_n_cheapest_dishes(self):
        """Показать N самых дешёвых блюд"""
        try:
            n = int(input("Введите количество блюд (N): "))
            
            if n <= 0:
                print("Ошибка: введите положительное число")
                return
            
            with self.conn.cursor(cursor_factory=RealDictCursor) as cur:
                cur.execute("""
                    SELECT d.title, d.price, c.name as category
                    FROM dish d
                    LEFT JOIN category c ON d.category_id = c.id
                    ORDER BY d.price
                    LIMIT %s
                """, (n,))
                
                dishes = cur.fetchall()
                
                print(f"\n--- {n} САМЫХ ДЕШЁВЫХ БЛЮД ---")
                for dish in dishes:
                    print(f"{dish['title']} - {dish['price']:.2f} руб. - {dish['category']}")
                print()
                
        except ValueError:
            print("Ошибка: введите целое число")
        except Exception as e:
            print(f"Ошибка при получении блюд: {e}")
    
    def show_categories_with_count(self):
        """Показать категории и количество блюд в каждой"""
        try:
            with self.conn.cursor(cursor_factory=RealDictCursor) as cur:
                cur.execute("""
                    SELECT c.name, COUNT(d.id) as dish_count
                    FROM category c
                    LEFT JOIN dish d ON c.id = d.category_id
                    GROUP BY c.id, c.name
                    ORDER BY c.name
                """)
                
                categories = cur.fetchall()
                
                print("\n--- КАТЕГОРИИ И КОЛИЧЕСТВО БЛЮД ---")
                for category in categories:
                    print(f"{category['name']} - {category['dish_count']} блюд")
                print()
                
        except Exception as e:
            print(f"Ошибка при получении категорий: {e}")
    
    def show_menu(self):
        """Отображение главного меню"""
        while True:
            print("\n=== МЕНЮ РЕСТОРАНА ===")
            print("1. Показать всё меню")
            print("2. Показать блюда в ценовом диапазоне")
            print("3. Поиск по началу названия")
            print("4. Показать N самых дешёвых блюд")
            print("5. Категории и количество блюд")
            print("6. Выход")
            
            choice = input("Выберите пункт меню (1-6): ").strip()
            
            if choice == '1':
                self.show_all_menu()
            elif choice == '2':
                self.show_dishes_in_price_range()
            elif choice == '3':
                self.search_by_prefix()
            elif choice == '4':
                self.show_n_cheapest_dishes()
            elif choice == '5':
                self.show_categories_with_count()
            elif choice == '6':
                print("До свидания!")
                break
            else:
                print("Неверный выбор. Попробуйте снова.")
    
    def close(self):
        """Закрытие соединения с базой данных"""
        if self.conn:
            self.conn.close()
            print("Соединение с базой данных закрыто")

def main():
    """Главная функция приложения"""
    db = None
    try:
        db = RestaurantMenuDB()
        db.show_menu()
    except KeyboardInterrupt:
        print("\n\nПрограмма прервана пользователем")
    except Exception as e:
        print(f"Критическая ошибка: {e}")
    finally:
        if db:
            db.close()

if __name__ == "__main__":
    main()

Успешное подключение к базе данных
База данных инициализирована успешно

=== МЕНЮ РЕСТОРАНА ===
1. Показать всё меню
2. Показать блюда в ценовом диапазоне
3. Поиск по началу названия
4. Показать N самых дешёвых блюд
5. Категории и количество блюд
6. Выход
Выберите пункт меню (1-6): 2
Введите минимальную цену: 1000
Введите максимальную цену: 300000

--- БЛЮДА В ДИАПАЗОНЕ ЦЕН 1000.0-300000.0 руб. ---
Стейк рибай - 1200.00 руб. - Горячее


=== МЕНЮ РЕСТОРАНА ===
1. Показать всё меню
2. Показать блюда в ценовом диапазоне
3. Поиск по началу названия
4. Показать N самых дешёвых блюд
5. Категории и количество блюд
6. Выход
Выберите пункт меню (1-6): 2
Введите минимальную цену: 10000000
Введите максимальную цену: 130000000
Блюд в указанном диапазоне цен не найдено

=== МЕНЮ РЕСТОРАНА ===
1. Показать всё меню
2. Показать блюда в ценовом диапазоне
3. Поиск по началу названия
4. Показать N самых дешёвых блюд
5. Категории и количество блюд
6. Выход
Выберите пункт меню (1-6): 4
Введите количество б